# Notebook 01: Exploratory Data Analysis (EDA)
## Daily Energy Consumption Forecasting Using Time Series Analysis

This notebook explores the preprocessed daily energy consumption dataset, analyzing:
1. Historical consumption trajectories (2006–2010)
2. Day-of-week patterns and weekend vs. weekday energy dynamics
3. Monthly and seasonal variations (winter heating vs. summer cooling)
4. Autocorrelation (ACF) and partial autocorrelation (PACF)
5. Incomplete days and measurement coverage distributions

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

### 1. Load Preprocessed Daily Dataset
Data has already been converted from minute-level active power (kW) to daily energy ($E = \sum P / 60\text{ kWh}$).

In [ ]:
DATA_PATH = '../data/processed/daily_energy.csv'
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Total days: {len(df)}")
print(f"Qualified eligible days (>=95% coverage): {df['is_eligible'].sum()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

### 2. Daily Energy Consumption Over Time
Plotting the full 47-month historical series alongside a 30-day centered moving average.

In [ ]:
eligible_df = df[df['is_eligible']].copy()
eligible_df['ma30'] = eligible_df['daily_energy_kwh'].rolling(30, center=True).mean()

plt.figure(figsize=(14, 5))
plt.plot(eligible_df['date'], eligible_df['daily_energy_kwh'], color='#0284c7', alpha=0.5, label='Daily Energy (kWh)')
plt.plot(eligible_df['date'], eligible_df['ma30'], color='#dc2626', linewidth=2, label='30-Day Moving Average')
plt.title('Daily Household Electricity Consumption (Dec 2006 - Nov 2010)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Energy Consumption (kWh)')
plt.legend()
plt.show()

### 3. Day of Week Consumption Analysis
Evaluating whether weekend routines generate higher domestic appliance usage.

In [ ]:
eligible_df['weekday'] = eligible_df['date'].dt.day_name()
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=eligible_df, x='weekday', y='daily_energy_kwh', order=order, palette='crest', ax=ax1)
ax1.set_title('Mean Daily Consumption by Weekday')
ax1.set_ylabel('Mean kWh')
ax1.tick_params(axis='x', rotation=30)

eligible_df['is_weekend'] = eligible_df['date'].dt.dayofweek.isin([5, 6]).map({True: 'Weekend', False: 'Weekday'})
sns.boxplot(data=eligible_df, x='is_weekend', y='daily_energy_kwh', palette='Blues', ax=ax2)
ax2.set_title('Weekday vs. Weekend Distribution')
ax2.set_ylabel('kWh')
plt.tight_layout()
plt.show()

### 4. Monthly & Annual Seasonality
Examines how heating in winter contrasts with lower consumption in summer months.

In [ ]:
eligible_df['month'] = eligible_df['date'].dt.month
eligible_df['year'] = eligible_df['date'].dt.year

plt.figure(figsize=(12, 4.5))
sns.boxplot(data=eligible_df, x='month', y='daily_energy_kwh', palette='mako')
plt.title('Monthly Distribution of Daily Energy Consumption', fontsize=13, fontweight='bold')
plt.xlabel('Month (1 = January, 12 = December)')
plt.ylabel('Daily Energy (kWh)')
plt.show()

### 5. Autocorrelation (ACF) Analysis
Demonstrates strong weekly seasonality at lag 7, lag 14, and lag 21.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(eligible_df['daily_energy_kwh'].dropna(), lags=45, ax=ax, color='#0284c7')
plt.title('Autocorrelation Function (ACF) - Lags up to 45 Days', fontsize=13, fontweight='bold')
plt.xlabel('Lag (Days)')
plt.ylabel('Correlation')
plt.show()